<a href="https://colab.research.google.com/github/realshubhamraut/CDAC-DBDA-coursework/blob/main/08.advanced-analytics-stats/assignments/final_case_study/case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### consumer lending risk insights through data-driven analytics

analyzing credit risk patterns to identify default drivers and improve lending decisions.

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import chi2_contingency, f, norm, t
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats import weightstats as ssw
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')

sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 50)

#### data loading

In [ ]:
app = pd.read_csv('../../../data/credit_risk_applicants.csv', encoding='latin-1')
meta = pd.read_csv('../../../data/credit_risk_metadata.csv', encoding='latin-1')
loans = pd.read_csv('../../../data/credit_risk_previous_loans.csv', encoding='latin-1')

#### dataset overview

In [ ]:
app.shape, loans.shape

In [ ]:
app.head(3)

In [ ]:
app.info()

#### data quality assessment

In [ ]:
missing_pct = (app.isnull().sum() / len(app) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].head(20)

In [ ]:
app['TARGET'].value_counts()

In [ ]:
default_rate = app['TARGET'].mean()
default_rate

#### data cleaning

In [ ]:
app['AGE_YEARS'] = -app['DAYS_BIRTH'] / 365
app['EMPLOYMENT_YEARS'] = -app['DAYS_EMPLOYED'] / 365
app['EMPLOYMENT_YEARS'] = app['EMPLOYMENT_YEARS'].apply(lambda x: np.nan if x > 100 else x)
app['CREDIT_INCOME_RATIO'] = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']

#### univariate analysis - target distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
app['TARGET'].value_counts().plot(kind='bar', ax=ax[0], color=['#66c2a5', '#fc8d62'])
ax[0].set_title('target distribution')
ax[0].set_xlabel('target')
ax[0].set_ylabel('count')
ax[0].set_xticklabels(['non-defaulter', 'defaulter'], rotation=0)

app['TARGET'].value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'])
ax[1].set_title('default rate')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

#### univariate analysis - numerical features

In [ ]:
num_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AGE_YEARS', 'EMPLOYMENT_YEARS']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    app[col].hist(bins=50, ax=axes[i], edgecolor='black', alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('frequency')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
app[num_cols].describe()

#### univariate analysis - categorical features

In [ ]:
cat_cols = ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    app[col].value_counts().plot(kind='barh', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('count')

plt.tight_layout()
plt.show()

#### bivariate analysis - income vs default

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

app.boxplot(column='AMT_INCOME_TOTAL', by='TARGET', ax=axes[0])
axes[0].set_title('income distribution by default status')
axes[0].set_xlabel('target')
axes[0].set_ylabel('income')
axes[0].set_xticklabels(['non-defaulter', 'defaulter'])
plt.suptitle('')

for target in [0, 1]:
    subset = app[app['TARGET'] == target]['AMT_INCOME_TOTAL']
    axes[1].hist(subset, bins=50, alpha=0.6, label=f'target={target}', edgecolor='black')
axes[1].set_title('income distribution overlay')
axes[1].set_xlabel('income')
axes[1].set_ylabel('frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

#### bivariate analysis - credit amount vs default

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

app.boxplot(column='AMT_CREDIT', by='TARGET', ax=axes[0])
axes[0].set_title('credit amount by default status')
axes[0].set_xlabel('target')
axes[0].set_ylabel('credit amount')
axes[0].set_xticklabels(['non-defaulter', 'defaulter'])
plt.suptitle('')

app.boxplot(column='CREDIT_INCOME_RATIO', by='TARGET', ax=axes[1])
axes[1].set_title('credit-to-income ratio by default status')
axes[1].set_xlabel('target')
axes[1].set_ylabel('ratio')
axes[1].set_xticklabels(['non-defaulter', 'defaulter'])
plt.suptitle('')

plt.tight_layout()
plt.show()

#### bivariate analysis - age vs default

In [ ]:
app['AGE_GROUP'] = pd.cut(app['AGE_YEARS'], bins=[0, 25, 35, 45, 55, 100], labels=['<25', '25-35', '35-45', '45-55', '55+'])
age_default = pd.crosstab(app['AGE_GROUP'], app['TARGET'], normalize='index') * 100

age_default.plot(kind='bar', stacked=False, figsize=(10, 5))
plt.title('default rate by age group')
plt.xlabel('age group')
plt.ylabel('percentage')
plt.legend(['non-defaulter', 'defaulter'])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### bivariate analysis - education vs default

In [ ]:
edu_default = pd.crosstab(app['NAME_EDUCATION_TYPE'], app['TARGET'], normalize='index') * 100

edu_default.plot(kind='barh', stacked=False, figsize=(10, 5))
plt.title('default rate by education level')
plt.xlabel('percentage')
plt.ylabel('education type')
plt.legend(['non-defaulter', 'defaulter'])
plt.tight_layout()
plt.show()

#### bivariate analysis - income type vs default

In [ ]:
income_default = pd.crosstab(app['NAME_INCOME_TYPE'], app['TARGET'], normalize='index') * 100

income_default.plot(kind='barh', stacked=False, figsize=(10, 5))
plt.title('default rate by income type')
plt.xlabel('percentage')
plt.ylabel('income type')
plt.legend(['non-defaulter', 'defaulter'])
plt.tight_layout()
plt.show()

#### bivariate analysis - gender vs default

In [ ]:
gender_default = pd.crosstab(app['CODE_GENDER'], app['TARGET'], normalize='index') * 100

gender_default.plot(kind='bar', stacked=False, figsize=(8, 5))
plt.title('default rate by gender')
plt.xlabel('gender')
plt.ylabel('percentage')
plt.legend(['non-defaulter', 'defaulter'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

#### bivariate analysis - external sources vs default

In [ ]:
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(ext_sources):
    app.boxplot(column=col, by='TARGET', ax=axes[i])
    axes[i].set_title(f'{col} by default status')
    axes[i].set_xlabel('target')
    axes[i].set_ylabel('score')
    axes[i].set_xticklabels(['non-defaulter', 'defaulter'])

plt.suptitle('')
plt.tight_layout()
plt.show()

#### correlation analysis

In [ ]:
corr_cols = ['TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AGE_YEARS', 'EMPLOYMENT_YEARS', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'CREDIT_INCOME_RATIO']
corr_matrix = app[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=1)
plt.title('correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix['TARGET'].sort_values(ascending=False)

#### multivariate analysis - income vs credit vs default

In [ ]:
plt.figure(figsize=(10, 6))
for target in [0, 1]:
    subset = app[app['TARGET'] == target]
    plt.scatter(subset['AMT_INCOME_TOTAL'], subset['AMT_CREDIT'], alpha=0.3, label=f'target={target}')
plt.title('income vs credit amount by default status')
plt.xlabel('income')
plt.ylabel('credit amount')
plt.legend()
plt.tight_layout()
plt.show()

#### previous loans analysis

In [ ]:
loans['NAME_CONTRACT_STATUS'].value_counts()

In [ ]:
loan_status_counts = loans.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS'].value_counts().unstack(fill_value=0)
app_merged = app.merge(loan_status_counts, left_on='SK_ID_CURR', right_index=True, how='left')
app_merged.fillna(0, inplace=True)

In [ ]:
if 'Refused' in app_merged.columns:
    app_merged.boxplot(column='Refused', by='TARGET', figsize=(8, 5))
    plt.title('previous refusals by default status')
    plt.suptitle('')
    plt.xlabel('target')
    plt.ylabel('number of refusals')
    plt.xticks([1, 2], ['non-defaulter', 'defaulter'])
    plt.tight_layout()
    plt.show()

#### hypothesis testing setup

In [ ]:
defaulters = app[app['TARGET'] == 1]
non_defaulters = app[app['TARGET'] == 0]

#### hypothesis 1: do defaulters have significantly lower income?

h0: mean income of defaulters >= mean income of non-defaulters  
h1: mean income of defaulters < mean income of non-defaulters  
test: independent samples t-test (one-tailed)

In [ ]:
income_defaulters = defaulters['AMT_INCOME_TOTAL'].dropna()
income_non_defaulters = non_defaulters['AMT_INCOME_TOTAL'].dropna()

t_stat, p_value_two = stats.ttest_ind(income_defaulters, income_non_defaulters, equal_var=False)
p_value_one = p_value_two / 2

np.mean(income_defaulters), np.mean(income_non_defaulters), t_stat, p_value_one

result: defaulters have significantly lower income than non-defaulters.

#### hypothesis 2: is default rate different across genders?

h0: default rate is same for males and females  
h1: default rate differs between genders  
test: two-proportion z-test

In [ ]:
gender_data = app[app['CODE_GENDER'].isin(['M', 'F'])]
male_default = gender_data[gender_data['CODE_GENDER'] == 'M']['TARGET'].sum()
female_default = gender_data[gender_data['CODE_GENDER'] == 'F']['TARGET'].sum()
male_total = (gender_data['CODE_GENDER'] == 'M').sum()
female_total = (gender_data['CODE_GENDER'] == 'F').sum()

count_arr = np.array([male_default, female_default])
nobs_arr = np.array([male_total, female_total])

z_stat, p_value = proportions_ztest(count_arr, nobs_arr)
male_default / male_total, female_default / female_total, z_stat, p_value

result: default rate is significantly different between genders.

#### hypothesis 3: is education level associated with default?

h0: education level and default are independent  
h1: education level and default are related  
test: chi-square test of independence

In [ ]:
contingency_table = pd.crosstab(app['NAME_EDUCATION_TYPE'], app['TARGET'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

chi2, p_value, dof

result: education level is significantly associated with default status.

#### hypothesis 4: do defaulters have higher credit-to-income ratio?

h0: mean credit-to-income ratio is same for both groups  
h1: defaulters have higher credit-to-income ratio  
test: independent samples t-test (one-tailed)

In [ ]:
ratio_defaulters = defaulters['CREDIT_INCOME_RATIO'].dropna()
ratio_non_defaulters = non_defaulters['CREDIT_INCOME_RATIO'].dropna()

t_stat, p_value_two = stats.ttest_ind(ratio_defaulters, ratio_non_defaulters, equal_var=False)
p_value_one = p_value_two / 2 if t_stat > 0 else 1 - p_value_two / 2

np.mean(ratio_defaulters), np.mean(ratio_non_defaulters), t_stat, p_value_one

result: defaulters have significantly higher credit-to-income ratio.

#### hypothesis 5: does ext_source_2 differ between defaulters and non-defaulters?

h0: mean ext_source_2 is same for both groups  
h1: mean ext_source_2 differs between groups  
test: independent samples t-test

In [ ]:
ext2_defaulters = defaulters['EXT_SOURCE_2'].dropna()
ext2_non_defaulters = non_defaulters['EXT_SOURCE_2'].dropna()

t_stat, p_value = stats.ttest_ind(ext2_defaulters, ext2_non_defaulters, equal_var=False)

np.mean(ext2_defaulters), np.mean(ext2_non_defaulters), t_stat, p_value

result: ext_source_2 is significantly different, with defaulters having lower scores.

#### hypothesis 6: is default rate different across income types?

h0: default rate is same across all income types  
h1: default rate differs across income types  
test: chi-square test of independence

In [ ]:
income_contingency = pd.crosstab(app['NAME_INCOME_TYPE'], app['TARGET'])
chi2, p_value, dof, expected = chi2_contingency(income_contingency)

chi2, p_value, dof

result: default rate significantly varies across income types.

#### hypothesis 7: do defaulters have lower external source scores?

h0: mean ext_source_3 is same for both groups  
h1: defaulters have lower ext_source_3  
test: independent samples t-test (one-tailed)

In [ ]:
ext3_defaulters = defaulters['EXT_SOURCE_3'].dropna()
ext3_non_defaulters = non_defaulters['EXT_SOURCE_3'].dropna()

t_stat, p_value_two = stats.ttest_ind(ext3_defaulters, ext3_non_defaulters, equal_var=False)
p_value_one = p_value_two / 2

np.mean(ext3_defaulters), np.mean(ext3_non_defaulters), t_stat, p_value_one

result: defaulters have significantly lower ext_source_3 scores.

#### hypothesis 8: variance in credit amount - defaulters vs non-defaulters

h0: variance in credit amount is same for both groups  
h1: variance differs between groups  
test: f-test for variance

In [ ]:
credit_defaulters = defaulters['AMT_CREDIT'].dropna()
credit_non_defaulters = non_defaulters['AMT_CREDIT'].dropna()

f_stat = np.var(credit_defaulters, ddof=1) / np.var(credit_non_defaulters, ddof=1)
df1 = len(credit_defaulters) - 1
df2 = len(credit_non_defaulters) - 1
p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), stats.f.sf(f_stat, df1, df2))

np.var(credit_defaulters, ddof=1), np.var(credit_non_defaulters, ddof=1), f_stat, p_value

result: variance in credit amount differs significantly between groups.

#### key insights summary

1. defaulters have significantly lower income compared to non-defaulters
2. default rates vary significantly between genders
3. education level is strongly associated with default risk
4. defaulters have higher credit-to-income ratios
5. external credit bureau scores are strong predictors of default
6. income type influences default probability
7. younger applicants tend to have higher default rates
8. previous loan refusals correlate with future defaults

#### business recommendations

1. implement stricter income verification for low-income applicants
2. adjust credit limits based on credit-to-income ratio thresholds
3. incorporate external source scores as mandatory screening criteria
4. develop education-based risk scoring models
5. apply differential pricing based on identified risk factors
6. enhance documentation requirements for high-risk segments
7. consider age-adjusted credit policies
8. flag applicants with previous refusals for additional review